# Cluster of Computers: Webserver and MongoDB Backend

## Import the FABlib Library


In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

## Building A Two-Node Cluster

In [ ]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site
nodesReq = 2
coresReq = 4
ramReq = 4

totalCoreAvail = nodesReq * coresReq
totalRamAvail = nodesReq * ramReq

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail and ram >= totalRamAvail:
        usableSite.append(site)

print(usableSite)

In [ ]:
slice = fablib.new_slice(name="BigCluster")
siteName = 'TOKY'
network_name = 'ramnet'

# Network

net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    node = slice.add_node(name=f"node{i}", site=siteName, disk=20, image='default_ubuntu_22')
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)
    
slice.submit()          # blocking by default

In [ ]:
import time

while True:
    time.sleep(10)
    slice.update()
    print("Slice state:", slice.get_state())
    print("Slice stable:", slice.isStable())
    if node.get_management_ip() != None:
        for node in slice.get_nodes():
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

In [ ]:
from ipaddress import IPv4Network

slice = fablib.get_slice(name="BigCluster")

for i in range(1, nodesReq + 1):
    node = slice.get_node(name=f"node{i}")
    iface = node.get_interface(network_name=network_name)

    iface.ip_link_up()
    iface.ip_addr_add(
        addr=f"192.168.1.{i}",
        subnet=IPv4Network("192.168.1.0/24")
    )

In [ ]:
for i in range(1,nodesReq):
    src = slice.get_node(name="node" + str(i))
    for j in range(i + 1,nodesReq + 1):
        des = slice.get_node(name="node" + str(j))           
        des_addr = des.get_interface(network_name=network_name).get_ip_addr()
        print(f"{src.get_name()} is pinging {des.get_name()} at {des_addr} ========")
        stdout, stderr = src.execute(f'ping -c 2 {des_addr}')    

In [ ]:
from pathlib import Path

web_node = slice.get_node("node1")
db_node = slice.get_node("node2")

web_stdout, web_stderr = web_node.execute(
    "python3 -c 'import sys; print(sys.executable)'",
    quiet=True
)
db_stdout, db_stderr = db_node.execute(
    "python3 -c 'import sys; print(sys.executable)'",
    quiet=True
)

slice_key = fablib.get_default_slice_key()["slice_private_key_file"]

inventory = f"""all:
  vars:
    ansible_become: true
    mongo_admin_user: "admin"
    mongo_admin_password: "admin123"
    mongo_app_db: "demoapp"
    mongo_app_user: "demoapp"
    mongo_app_password: "demo123"

  children:
    webservers:
      hosts:
        web_node:
          ansible_host: "{web_node.get_management_ip()}"
          ansible_user: "{web_node.get_username()}"
          ansible_ssh_private_key_file: "{slice_key}"
          ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
          ansible_python_interpreter: "{web_stdout.strip()}"
          private_ip: "192.168.1.1"
          mongodb_private_ip: "192.168.1.2"

    dbservers:
      hosts:
        db_node:
          ansible_host: "{db_node.get_management_ip()}"
          ansible_user: "{db_node.get_username()}"
          ansible_ssh_private_key_file: "{slice_key}"
          ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"
          ansible_python_interpreter: "{db_stdout.strip()}"
          private_ip: "192.168.1.2"
"""

Path("playbook").mkdir(exist_ok=True)
Path("playbook/inventory.yml").write_text(inventory)

print("Wrote playbook/inventory.yml")
print(inventory)

In [ ]:
!ansible-playbook -i playbook/inventory.yml playbook/playbook.yml

In [ ]:
stdout, stderr = web_node.execute("curl 127.0.0.1:5000", quiet=True);
print(stdout)

In [ ]:
slice.delete()